# Task 6: Locked Evaluation Generation & Blind Pairwise Packaging (`align-generate`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__)


In [ ]:
# 1. Run 3-prompt sanity checks FIRST with locked settings (greedy, max_new_tokens=512, repetition_penalty=1.0)
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_alpaca_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_hh/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_hh_dpo_hh.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_alpaca_dpo_uf.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3


In [ ]:
# 2. Full 300-prompt generations with locked settings (greedy, max_new_tokens=512, repetition_penalty=1.0)
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/sft/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_sft.json \
  --generator_name qwen2.5-0.5b-sft \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_hh/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_dpo_hh.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_hh/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_dpo_hh.json \
  --generator_name qwen2.5-0.5b-dpo-hh \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_dpo_uf.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_dpo_uf.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16
